# 03 · Ocena polityki z doświadczenia

**Cel.** Oblicz zwrot po zakończeniu epizodu, wykonaj aktualizację po jednym przejściu i porównaj oba sposoby oceny tej samej polityki.

Pracujemy na planszy 4×4 z poprzedniej części. Wiersze od góry to `SFFF / FHFH / FFFH / HFFG`. `S` oznacza start, `F` lód, `H` dziurę, a `G` cel. Nagroda za wejście do celu wynosi 1. Pozostałe ruchy dają 0. Wejście do celu lub dziury kończy epizod. Przy poślizgu realizuje się komenda albo jeden z dwóch kierunków prostopadłych, każdy z prawdopodobieństwem $1/3$. Ruch za brzeg pozostawia agenta na miejscu.

Polityka $\pi$ wybiera **prawo w stanie 14**, **dół w stanie 10**, a na pozostałych polach losuje cztery komendy równomiernie. Oceniamy tę politykę przez cały eksperyment. $V^\pi(s)$ to oczekiwany zdyskontowany zwrot, gdy zaczynamy w $s$ i wykonujemy komendy zgodne z $\pi$.

Ustaw plik `srodowiska.py` obok notebooka. Uruchamiaj komórki po kolei. Po każdym `TODO` jest mały test z wynikiem, który można sprawdzić na kartce.

In [ ]:
from math import isclose, sqrt
from random import Random
from statistics import mean, stdev
from typing import NamedTuple

from srodowiska import FrozenLake

GAMMA = 0.95
LEWO, DOL, PRAWO, GORA = range(4)
env = FrozenLake(slippery=True)
live_states = tuple(s for s in range(env.n_states) if not env.terminal(s))


class Transition(NamedTuple):
    state: int
    action: int
    reward: float
    next_state: int
    done: bool


def choices(state):
    """Możliwe komendy polityki, każda z jednakową szansą w tej krotce."""
    if state == 14:
        return (PRAWO,)
    if state == 10:
        return (DOL,)
    return env.actions


episode_example = [
    Transition(14, PRAWO, 0.0, 10, False),
    Transition(10, DOL, 0.0, 14, False),
    Transition(14, PRAWO, 1.0, 15, True),
]

## Od trzech gałęzi do jednej próbki

Agent zaczyna w polu 14. Stara tablica zawiera $V(14)=0{,}60$ i $V(10)=0{,}40$. Przy dyskoncie $\gamma=0{,}95$ aktualizacja z pełnego modelu daje

$$
\tfrac13(0+0{,}95\cdot0{,}40)+\tfrac13(0+0{,}95\cdot0{,}60)+\tfrac13(1)=0{,}65.
$$

Trzy składniki odpowiadają ruchowi do 10, zatrzymaniu przy brzegu i wejściu do celu. W jednej rozgrywce obserwujemy następujący epizod.

| Stan | Komenda | Następny stan | Nagroda | Koniec |
|---:|---|---:|---:|---|
| 14 | prawo | 10 | 0 | nie |
| 10 | dół | 14 | 0 | nie |
| 14 | prawo | 15 | 1 | tak |

**Na kartce.** Oblicz całą zdyskontowaną sumę nagród od pierwszego wejścia w stan 14. Następnie odegraj aktualizację po pierwszym przejściu, gdy znasz tylko nową pozycję 10 i bieżące $V(10)$. Która liczba jest dostępna wcześniej i od jakich danych zależy?

## 1 · Zwroty dla wszystkich wizyt

Monte Carlo (MC) wyznacza cel z pełnego, zakończonego epizodu. Zwrot od chwili $t$ można policzyć od końca przez rekurencję

$$
G_t=R_{t+1}+\gamma G_{t+1}, \qquad G_T=0.
$$

`episode[t]` zawiera nagrodę $R_{t+1}$ i stan $S_t$. Do `discounted_returns` wpisz obliczanie zwrotów od ostatniego przejścia do pierwszego. Zwrócona lista zachowuje kolejność wizyt w epizodzie.

In [ ]:
def discounted_returns(episode, gamma):
    """Zwróć listę G_t w tej samej kolejności co przejścia."""
    # TODO 1. Przygotuj listę długości len(episode) i ustaw G = 0.
    # Przejdź od ostatniego indeksu do zera. Oblicz G = nagroda + gamma * G.
    # Wpisz G pod danym indeksem i po pętli zwróć listę.
    raise NotImplementedError('Uzupełnij obliczanie zwrotów')

In [ ]:
returns = discounted_returns(episode_example, GAMMA)
assert len(returns) == 3
assert all(isclose(g, expected) for g, expected in zip(returns, (0.9025, 0.95, 1.0)))
assert discounted_returns([], GAMMA) == []
print('Zwroty od poszczególnych wizyt:', returns)

## 2 · Monte Carlo pierwszej wizyty

W obrębie jednego epizodu stan 14 pojawia się dwa razy. Metoda **pierwszej wizyty** dołącza jedną próbkę zwrotu od chronologicznie pierwszej wizyty w danym stanie. Dla 14 jest to $0{,}9025$. Ostatnia wizyta dałaby $1$.

$$
N(s)\gets N(s)+1, \qquad V(s)\gets V(s)+\frac{G_t-V(s)}{N(s)}.
$$

To ta sama aktualizacja średniej, którą stosowaliśmy w ćwiczeniu o bandytach. Przejdź chronologicznie po parach przejście i zwrot. Przechowuj `seen` osobno dla każdego epizodu. `values` i `counts` to listy indeksowane numerem pola. Zaktualizuj je w miejscu.

In [ ]:
def mc_first_visit(values, counts, episode, gamma):
    """Wykonaj aktualizacje MC pierwszej wizyty dla zakończonego epizodu."""
    # TODO 2. Oblicz zwroty i zacznij od pustego zbioru seen.
    # Dla każdej pary (przejście, zwrot) sprawdź, czy stan już był w seen.
    # Przy pierwszej wizycie dodaj stan do seen, zwiększ licznik i popraw średnią.
    raise NotImplementedError('Uzupełnij aktualizację MC pierwszej wizyty')

In [ ]:
values = [0.0] * env.n_states
counts = [0] * env.n_states
mc_first_visit(values, counts, episode_example, GAMMA)
assert counts[14] == counts[10] == 1
assert isclose(values[14], 0.9025) and isclose(values[10], 0.95)
mc_first_visit(values, counts, [Transition(14, PRAWO, 1.0, 15, True)], GAMMA)
assert counts[14] == 2 and isclose(values[14], (0.9025 + 1.0) / 2)
print('MC poprawnie zatrzymuje pierwszą wizytę i uśrednia wynik przez epizody.')

## 3 · Jedna aktualizacja TD(0)

TD to *temporal difference*, czyli metoda różnic czasowych. Jej cel po jednym przejściu łączy rzeczywistą nagrodę i bieżące oszacowanie wartości następnego stanu. W stanie końcowym dalszy wkład wynosi 0.

$$
Y_t=R_{t+1}+\gamma\,\mathbf 1_{\{\text{epizod trwa}\}}V(S_{t+1}),\qquad
\delta_t=Y_t-V(S_t),\qquad V(S_t)\gets V(S_t)+\alpha\delta_t.
$$

Użycie bieżącego oszacowania w celu to *bootstrapowanie*. Przy ruchu 14 → 10 cel wynosi $0{,}38$. Jeśli $\alpha=0{,}10$, nowa wartość to $0{,}60+0{,}10(0{,}38-0{,}60)=0{,}578$. W eksperymencie wielokrotnym wybieramy $\alpha=N(s)^{-0{,}6}$ po zwiększeniu licznika wizyt. Ten malejący krok ogranicza wpływ kolejnych pojedynczych próbek. Wartość parametru i sposób liczenia wizyt są częścią opisu porównania.

In [ ]:
def td0_update(values, counts, transition, gamma, alpha=None):
    """Wykonaj aktualizację w miejscu i zwróć użyty cel Y_t."""
    # TODO 3. Odczytaj stan, nagrodę, następny stan i informację o końcu.
    # Cel to r dla terminala lub r + gamma * V(następny stan) w innym przypadku.
    # Zwiększ licznik. Użyj alpha, a przy None wybierz krok counts[stan] ** (-0.6).
    # Zaktualizuj bieżący stan o krok razy różnica między celem a oszacowaniem.
    raise NotImplementedError('Uzupełnij aktualizację TD(0)')

In [ ]:
values = [0.0] * env.n_states
counts = [0] * env.n_states
values[14], values[10], values[15] = 0.60, 0.40, 99.0
first_target = td0_update(values, counts, episode_example[0], GAMMA, alpha=0.10)
assert isclose(first_target, 0.38) and isclose(values[14], 0.578)
terminal_target = td0_update(values, counts, episode_example[-1], GAMMA, alpha=0.10)
assert isclose(terminal_target, 1.0) and isclose(values[14], 0.6202)
assert counts[14] == 2
print('TD(0) pomija dalszą wartość po zakończeniu epizodu.')

## 4 · Wspólne doświadczenie, wspólny punkt odniesienia

Porównujemy metody, używając tej samej listy przejść w każdym epizodzie. MC widzi wyniki po zakończeniu epizodu. TD czyta przejścia w kolejności i stosuje aktualizację po każdym z nich. Obie tablice startują od zera. W porównaniu badamy błąd wszystkich stanów nieterminalnych, więc początek epizodu losujemy równomiernie spośród tych pól. Po pierwszej komendzie obie metody nadal korzystają z tej samej polityki $\pi$.

Wartość referencyjna pochodzi z iteracyjnej oceny polityki z pełnego modelu. To pomoc do pomiaru błędu. Funkcje MC i TD pobierają przejścia z epizodów. Na planszy z tą polityką z każdego pola istnieje ścieżka o dodatnim prawdopodobieństwie do terminala. Wprowadzony limit długości to kontrola wykonania. Jeśli epizod go przekroczy, kod zgłosi błąd, ponieważ w tym ćwiczeniu MC wymaga pełnego zwrotu.

In [ ]:
def model_reference(gamma):
    """Policz V polityki przez synchroniczne aktualizacje z pełnego modelu."""
    v = [0.0] * env.n_states
    for _ in range(5000):
        next_v = [0.0] * env.n_states
        for s in live_states:
            next_v[s] = mean(
                sum(p * (r + gamma * (0.0 if done else v[sp]))
                    for p, sp, r, done in env.outcomes(s, a))
                for a in choices(s)
            )
        if max(abs(a - b) for a, b in zip(v, next_v)) < 1e-12:
            return next_v
        v = next_v
    raise RuntimeError('Ocena modelowa nie osiągnęła zadanej tolerancji')


def fixed_command(state, rng):
    return rng.choice(choices(state))


def sample_episode(rng, cap=10_000, command=fixed_command, start=None):
    s = rng.choice(live_states) if start is None else start
    episode = []
    for _ in range(cap):
        a = command(s, rng)
        sp, r, done = env.sample(s, a, rng)
        episode.append(Transition(s, a, r, sp, done))
        if done:
            return episode
        s = sp
    raise RuntimeError('Osiągnięto limit, zanim epizod dotarł do terminala')


def rmse(values, reference):
    """Pierwiastek ze średniego kwadratu błędu po stanach nieterminalnych."""
    return sqrt(mean((values[s] - reference[s]) ** 2 for s in live_states))


def run(seed, checkpoints, reference):
    rng = Random(seed)
    mc_values, td_values = ([0.0] * env.n_states for _ in range(2))
    mc_counts, td_counts = ([0] * env.n_states for _ in range(2))
    measurements = {}
    transitions = 0
    for n in range(1, max(checkpoints) + 1):
        episode = sample_episode(rng)
        transitions += len(episode)
        mc_first_visit(mc_values, mc_counts, episode, GAMMA)
        for transition in episode:
            td0_update(td_values, td_counts, transition, GAMMA)
        if n in checkpoints:
            measurements[n] = {
                'mc': rmse(mc_values, reference),
                'td': rmse(td_values, reference),
                'transitions': transitions,
                'min_visits': min(mc_counts[s] for s in live_states),
            }
    return measurements


reference = model_reference(GAMMA)
assert all(reference[s] == 0.0 for s in range(env.n_states) if env.terminal(s))
assert 0 < reference[14] < 1 and 0 < reference[10] < 1
print(f'Wartość odniesienia po ocenie polityki: V(14) = {reference[14]:.4f}, V(10) = {reference[10]:.4f}')

In [ ]:
SEEDS = (7, 19, 31, 43, 61)
CHECKPOINTS = (25, 100, 400, 1600)
results = [run(seed, CHECKPOINTS, reference) for seed in SEEDS]

print('Epizody   Przejścia średnio   RMSE MC (śr. ± odch.)   RMSE TD (śr. ± odch.)   Min. MC')
for n in CHECKPOINTS:
    rows = [result[n] for result in results]
    mc = [row['mc'] for row in rows]
    td = [row['td'] for row in rows]
    steps = mean(row['transitions'] for row in rows)
    visits = min(row['min_visits'] for row in rows)
    print(f'{n:7} {steps:18.0f}              {mean(mc):.4f} ± {stdev(mc):.4f}'
          f'         {mean(td):.4f} ± {stdev(td):.4f}           {visits:5}')

### Zapisz wniosek

1. Zanim uruchomisz ostatnią komórkę, przewidź, przy jakiej liczbie epizodów bootstrapowanie może pomóc. Po uruchomieniu sprawdź swoją hipotezę na tabeli.
2. Wskaż po jednym wierszu z wczesnym i późnym pomiarem. Wyjaśnij, co w tym eksperymencie oznacza mniejszy RMSE, i porównaj różnicę z odchyleniem między pięcioma powtórzeniami.
3. `Min. MC` podaje najmniejszą liczbę pierwszych wizyt MC w jakimkolwiek polu i powtórzeniu. Co wynika z wiersza, w którym jest 0? Co by się zmieniło, gdyby startować zawsze w polu 0?
4. Wyjaśnij mechanizm na dwóch liczbach z karty. Cel MC dla pierwszej wizyty w 14 to $0{,}9025$. Cel TD po pierwszym przejściu to $0{,}38$. Dlaczego różnią się pomimo wspólnej polityki i tej samej rozgrywki?

### Dalej · od oceny do wyboru

Do wyboru komendy przyda się $Q^\pi(s,a)$, czyli oczekiwany zwrot po wskazanej pierwszej komendzie, a później zgodnie z polityką. Oszacowanie $Q$ pozwoli porównać komendy z tego samego pola. Następny moduł pokazuje, jak aktualizować tablicę $Q$ po każdym ruchu. Ćwiczenie dodatkowe poniżej robi to na podstawie zakończonych epizodów.

## Dodatkowe · sterowanie przez Monte Carlo

W ocenie polityki zbieraliśmy doświadczenie według ustalonej $\pi$. Teraz po każdym epizodzie poprawiamy oszacowanie $Q(s,a)$. W następnym epizodzie wybieramy akcje według reguły $\varepsilon$-zachłannej względem tej świeżej tablicy. Z szansą $\varepsilon$ losujemy jedną z czterech komend. W pozostałych przypadkach losujemy tylko wśród komend o największej wartości $Q(s,a)$.

Każda komenda ma w tej polityce szansę co najmniej $\varepsilon/4$. Dlatego nazywamy ją $\varepsilon$-miękką. Zbieranie danych i wybór dalszych komend w uczonej polityce podlegają tej samej regule. Tablicę aktualizujemy po zakończeniu przebiegu. Przy stałym $\varepsilon>0$ polityka zbierająca dane stale eksploruje. Ten eksperyment uczy się ze skończonej liczby przebiegów, a wynik odsłoniętej polityki zachłannej oceniamy empirycznie.

**Na kartce.** Dla stanu 14 załóż $Q(14,\textrm{prawo})=0{,}6$, a pozostałe wartości równe 0. Policz prawdopodobieństwo wybrania „w prawo” przy $\varepsilon=0{,}20$. W epizodzie 14 → 10 → 14 → 15 wskaż, z którego zwrotu skorzysta aktualizacja $Q(14,\textrm{prawo})$ przy regule pierwszej wizyty.

Zmodyfikuj wcześniejszą aktualizację pierwszej wizyty. Kluczem zbioru `seen` staje się para `(stan, akcja)`, a wpisy znajdują się w `q[s][a]` i `counts[s][a]`.

In [ ]:
def mc_q_first_visit(q, counts, episode, gamma):
    """Zaktualizuj średni zwrot dla pierwszej wizyty każdej pary stan–akcja."""
    # TODO dodatkowe. Zastosuj wcześniejszy algorytm z kluczem (stan, akcja).
    raise NotImplementedError('Uzupełnij aktualizację MC dla Q')

In [ ]:
def empty_q(value):
    return [[value] * len(env.actions) for _ in range(env.n_states)]


def epsilon_greedy(scores, rng, epsilon):
    if rng.random() < epsilon:
        return rng.choice(env.actions)
    best = max(scores)
    return rng.choice([a for a in env.actions if scores[a] == best])


def train_mc_control(seed, count, epsilon):
    rng, q, visits = Random(seed), empty_q(0.0), empty_q(0)

    def command(state, rollout_rng):
        return epsilon_greedy(q[state], rollout_rng, epsilon)

    for _ in range(count):
        episode = sample_episode(rng, command=command)
        mc_q_first_visit(q, visits, episode, GAMMA)
    return q, visits


def evaluate_greedy(q, seed, count):
    rng = Random(seed)

    def command(state, rollout_rng):
        return epsilon_greedy(q[state], rollout_rng, 0.0)

    return mean(sample_episode(rng, command=command, start=env.start)[-1].reward
                for _ in range(count))


toy_q, toy_counts = empty_q(0.0), empty_q(0)
mc_q_first_visit(toy_q, toy_counts, episode_example, GAMMA)
assert toy_counts[14][PRAWO] == 1 and isclose(toy_q[14][PRAWO], 0.9025)
assert toy_counts[10][DOL] == 1 and isclose(toy_q[10][DOL], 0.95)
assert sum(map(sum, toy_counts)) == 2


q, visits = train_mc_control(seed=73, count=6000, epsilon=0.20)
directions = ('lewo', 'dół', 'prawo', 'góra')
print('Najlepsza oceniona komenda w 14:', directions[max(env.actions, key=lambda a: q[14][a])])
print('Niezależna ocena zachłannej polityki od startu. Udział sukcesów:',
      evaluate_greedy(q, seed=101, count=400))
print('Najmniejsza liczba pierwszych wizyt dla jednej pary stan–akcja:',
      min(visits[s][a] for s in live_states for a in env.actions))

**Po uruchomieniu.** Zmień ziarno uczenia i powtórz ocenę zachłannej polityki. Zapisz, czy zmieniła się najlepsza komenda w 14. Wskaż co najmniej jedną przyczynę, dla której pojedynczy odsetek sukcesów może się zmienić. Jak zmieniłby się algorytm, gdyby aktualizacja $Q$ miała zajść po każdym przejściu?

## Dodatkowe · ocena innej polityki metodą Monte Carlo

Rozważ osobne ćwiczenie o jednym stanie. Każda z czterech akcji kończy epizod. „Prawo” daje nagrodę 1, pozostałe akcje dają 0. Polityka zbierająca dane $b$ losuje akcje równomiernie, a oceniana polityka $\pi$ wybiera zawsze „prawo”. Taka ocena z danych innej polityki to **ocena off-policy**.

Każdy zwrot $G_i$ mnożymy przez iloraz prawdopodobieństw $\rho_i=\pi(a_i\mid s)/b(a_i\mid s)$. W tym przykładzie waga dla „prawo” wynosi 4, dla pozostałych zaobserwowanych akcji 0. Warunek pokrycia wymaga, by każda akcja możliwa według $\pi$ miała dodatnią szansę w $b$.

$$
\widehat V_{\rm zwykły}=\frac{\sum_i \rho_i G_i}{n},\qquad
\widehat V_{\rm ważony}=\frac{\sum_i \rho_i G_i}{\sum_i \rho_i}.
$$

**Na kartce.** W sześciu epizodach zaobserwowano komendy `lewo, prawo, góra, dół, prawo, lewo`. Wypisz sześć wag. Policz oba oszacowania i zastanów się, co stanie się z drugim, gdy w krótkiej próbce zabraknie „prawo”.

In [ ]:
def off_policy_estimates(commands):
    weights = [4.0 if action == PRAWO else 0.0 for action in commands]
    returns = [float(action == PRAWO) for action in commands]
    weighted_sum = sum(w * g for w, g in zip(weights, returns))
    total_weight = sum(weights)
    ordinary = weighted_sum / len(commands)
    weighted = weighted_sum / total_weight if total_weight else None
    return weights, ordinary, weighted


commands = (LEWO, PRAWO, GORA, DOL, PRAWO, LEWO)
weights, ordinary, weighted = off_policy_estimates(commands)
print(f'Wagi {weights}. Estymator zwykły {ordinary:.3f}, ważony {weighted:.3f}.')
assert isclose(ordinary, 4.0 / 3.0) and weighted == 1.0
assert off_policy_estimates((LEWO, GORA, DOL))[2] is None